# Udaplay Project
## Part 02 - Agent

Now we build **UdaPlay**, an agent that answers game questions.

For every question it goes through the same simple steps:

```
retrieve_game  ->  evaluate_retrieval  ->  (game_web_search, only if needed)  ->  final answer
```

1. **retrieve_game**: search our own vector database from Part 1.
2. **evaluate_retrieval**: a second LLM acts as a judge and decides whether those results answer the question.
3. **game_web_search**: only if the judge says *not useful*, search the web.
4. **Answer**: a structured answer with a **source** and a **confidence** level.

The agent runs on the course's `StateMachine` (`lib/agents.py`), and it keeps conversation history per session. That's why follow-up questions work.

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import json
from typing import Literal

import chromadb
from chromadb.utils import embedding_functions
from pydantic import BaseModel, Field
from openai import OpenAI

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, AIMessage
from lib.tooling import tool

In [3]:
import os
from dotenv import load_dotenv, find_dotenv

# find_dotenv() finds the .env file in this folder (or a parent folder);
# override=True makes the values in .env win over anything already set.
load_dotenv(find_dotenv(), override=True)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
# Web search uses OpenAI's built-in web search tool with the same key,
# so a TAVILY_API_KEY is not needed.
print("Environment variables are set.")

Environment variables are set.


### Tools
Three tools. Each prints what it is doing, so you can follow the agent's steps in the output.

#### Retrieve Game Tool
Searches the vector DB and returns the 3 closest games.

In [4]:
# Load the database built in Part 1 (same path, same name, same embedding function)
chroma_client = chromadb.PersistentClient(path="chromadb")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
    model_name="text-embedding-ada-002",
)
collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)
print("Games in vector DB:", collection.count())


@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, PlayStation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    print(f"\n[STEP 1 - retrieve_game] query = {query!r}")
    results = collection.query(query_texts=[query], n_results=3)
    docs = []
    for meta, dist in zip(results["metadatas"][0], results["distances"][0]):
        docs.append({**meta, "distance": round(dist, 3)})
        print(f"   found: {meta['Name']} ({meta['Platform']}, {meta['YearOfRelease']})  distance={dist:.3f}")
    return docs

Games in vector DB: 15


#### Evaluate Retrieval Tool
An "LLM as judge": it reads the question and the retrieved games and returns a structured `EvaluationReport` (useful yes/no, confidence, reason).

In [5]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="True if the documents are enough to answer the question")
    confidence: Literal["high", "medium", "low"] = Field(description="How sure the judge is")
    description: str = Field(description="Short explanation of the decision")


@tool
def evaluate_retrieval(question: str, retrieved_docs: list[str]) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - confidence: high / medium / low
    - description: description about the evaluation result
    """
    print(f"\n[STEP 2 - evaluate_retrieval] question = {question!r}")
    judge = LLM(model="gpt-4o-mini", temperature=0)
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Only say useful=true if the documents directly contain the answer. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Query: {question}\n\nDocuments:\n" + "\n".join(f"- {d}" for d in retrieved_docs)
    )
    response = judge.invoke(prompt, response_format=EvaluationReport)
    report = EvaluationReport.model_validate_json(response.content)
    print(f"   useful={report.useful}  confidence={report.confidence}")
    print(f"   reason: {report.description}")
    return report.model_dump()

#### Game Web Search Tool
The fallback. It searches the web with OpenAI's built-in web search tool (same `voc-` key, no Tavily key needed) and returns the page URLs so the answer can cite them.

In [6]:
# Web search uses OpenAI's built-in web search tool (Responses API),
# through the same Vocareum key. It returns an answer plus the URLs it used.
web_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"), base_url=os.getenv("OPENAI_BASE_URL"))


@tool
def game_web_search(question: str) -> dict:
    """
    Web search: searches the web when the vector DB results aren't good enough
    args:
    - question: a question about game industry.

    Returns the search results, including each page's URL, so the answer can cite its source.
    """
    print(f"\n[STEP 3 - game_web_search] question = {question!r}")
    response = web_client.responses.create(
        model="gpt-4o-mini",
        tools=[{"type": "web_search_preview"}],
        tool_choice="required",  # always really search the web
        input=f"Answer this video game question using the web: {question}",
    )
    # Collect the source URLs the search cited
    sources = []
    for item in response.output:
        if item.type == "message":
            for part in item.content:
                for ann in getattr(part, "annotations", None) or []:
                    url = getattr(ann, "url", None)
                    if url and url not in [src["url"] for src in sources]:
                        sources.append({"title": getattr(ann, "title", ""), "url": url})
    for src in sources:
        print(f"   source: {src['url']}")
    return {"answer": response.output_text, "sources": sources}

### Agent
The `Agent` class from `lib/agents.py` is a **state machine**:

```
entry -> message_prep -> llm_processor --(tool calls?)--> tool_executor -> llm_processor ... -> termination
```

The LLM picks tools according to the instructions below. Every question in the same `session_id` adds to one shared message history (short-term memory).

In [7]:
INSTRUCTIONS = """
You are UdaPlay, a research assistant for video games.

Follow these steps for EVERY question, in this order. NEVER skip steps 1 and 2,
even if you think the answer needs the web (for example, current news):
1. Call retrieve_game to search the internal game database.
   If the question refers to something earlier in the conversation (like "it" or "that game"),
   rewrite it into a full question first, using the conversation history.
2. Call evaluate_retrieval with the question and the retrieved documents.
3. ONLY if evaluate_retrieval says useful = false, call game_web_search.
   If useful = true, do NOT search the web.
4. Write the final answer in exactly this format:

**Answer:** <a clear, short answer in natural language>
**Source:** <"Internal database: <game name(s)>" or "Web: <URL(s)>" - always include URLs for web answers>
**Confidence:** <high / medium / low, with one short reason>
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0,
)
print(agent.workflow)

StateMachine(schema=['user_query', 'instructions', 'messages', 'current_tool_calls', 'total_tokens'])


In [8]:
def ask(question, session_id="demo"):
    """Ask the agent a question, then print a short report of what it did."""
    print("=" * 90)
    print("QUESTION:", question)
    print("=" * 90)
    run = agent.invoke(question, session_id=session_id)
    messages = run.get_final_state()["messages"]

    # Only look at messages from THIS question (after the last user message)
    last_user = max(i for i, m in enumerate(messages) if isinstance(m, UserMessage))
    tools_used = [
        call.function.name
        for m in messages[last_user:]
        if isinstance(m, AIMessage) and m.tool_calls
        for call in m.tool_calls
    ]
    print("\n--- REPORT ---")
    print("Tools used, in order:", " -> ".join(tools_used) or "none")
    print("Web search needed:", "game_web_search" in tools_used)
    print()
    print(messages[-1].content)
    print()
    return run

### Run the agent
**Query 1 (in the local dataset).** We expect: retrieve -> evaluate (useful) -> answer. No web search.

In [9]:
ask("When was Pokémon Gold and Silver released?");

QUESTION: When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor

[STEP 1 - retrieve_game] query = 'When was Pokémon Gold and Silver released?'


   found: Pokémon Gold and Silver (Game Boy Color, 1999)  distance=0.118
   found: Pokémon Ruby and Sapphire (Game Boy Advance, 2002)  distance=0.151
   found: Super Mario 64 (Nintendo 64, 1996)  distance=0.229
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor

[STEP 2 - evaluate_retrieval] question = 'When was Pokémon Gold and Silver released?'


   useful=True  confidence=high
   reason: The document explicitly states the 'YearOfRelease' for Pokémon Gold and Silver as 1999. This directly answers the query regarding the release date of the games.
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

--- REPORT ---
Tools used, in order: retrieve_game -> evaluate_retrieval
Web search needed: False

**Answer:** Pokémon Gold and Silver were released in 1999.  
**Source:** Internal database: Pokémon Gold and Silver  
**Confidence:** high, the document explicitly states the release year.



**Query 2 (in the local dataset).** Super Mario 64 is in our data.

In [10]:
ask("Which one was the first 3D platformer Mario game?");

QUESTION: Which one was the first 3D platformer Mario game?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor

[STEP 1 - retrieve_game] query = 'Which one was the first 3D platformer Mario game?'


   found: Super Mario 64 (Nintendo 64, 1996)  distance=0.123
   found: Super Mario World (Super Nintendo Entertainment System (SNES), 1990)  distance=0.144
   found: Super Smash Bros. Melee (GameCube, 2001)  distance=0.208
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor

[STEP 2 - evaluate_retrieval] question = 'Which one was the first 3D platformer Mario game?'


   useful=True  confidence=high
   reason: The document explicitly states that 'Super Mario 64' is a 'groundbreaking 3D platformer' released in 1996. This directly answers the query about which was the first 3D platformer Mario game, as 'Super Mario 64' is widely recognized as the first 3D platformer in the Mario franchise.
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

--- REPORT ---
Tools used, in order: retrieve_game -> evaluate_retrieval
Web search needed: False

**Answer:** The first 3D platformer Mario game is Super Mario 64.  
**Source:** Internal database: Super Mario 64  
**Confidence:** high, the document explicitly identifies Super Mario 64 as the first 3D platformer.



**Query 3 (NOT in the local dataset).** Mortal Kombat X isn't one of our 15 games, so the judge should say *not useful* and the agent should search the web and cite URLs.

In [11]:
ask("Was Mortal Kombat X released for PlayStation 5?");

QUESTION: Was Mortal Kombat X released for PlayStation 5?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor

[STEP 1 - retrieve_game] query = 'Was Mortal Kombat X released for PlayStation 5?'


   found: Marvel's Spider-Man 2 (PlayStation 5, 2023)  distance=0.214
   found: Marvel's Spider-Man (PlayStation 4, 2018)  distance=0.228
   found: Halo Infinite (Xbox Series X|S, 2021)  distance=0.228
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor

[STEP 2 - evaluate_retrieval] question = 'Was Mortal Kombat X released for PlayStation 5?'


   useful=False  confidence=high
   reason: The document provided does not contain any information regarding the release of Mortal Kombat X for PlayStation 5. Instead, it discusses a different game, Marvel's Spider-Man 2, which is unrelated to the query. Therefore, the documents are insufficient to answer whether Mortal Kombat X was released for PlayStation 5.
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor

[STEP 3 - game_web_search] question = 'Was Mortal Kombat X released for PlayStation 5?'


   source: https://www.playstation.com/en-us/games/mortal-kombat-x/?video=0&utm_source=openai
   source: https://www.youtube.com/watch?v=tqsw711ZuAk&utm_source=openai
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

--- REPORT ---
Tools used, in order: retrieve_game -> evaluate_retrieval -> game_web_search
Web search needed: True

**Answer:** Mortal Kombat X was originally released for PlayStation 4 on April 14, 2015, and is playable on PlayStation 5 through backward compatibility, but it was not specifically released for PS5.  
**Source:** Web: [PlayStation](https://www.playstation.com/en-us/games/mortal-kombat-x/?video=0&utm_source=openai), [YouTube](https://www.youtube.com/watch?v=tqsw711ZuAk&utm_source=openai)  
**Confidence:** high, the information is clearly stated on the official PlayStation website.



**Query 4 (local).** Publisher question from the project scenario.

In [12]:
ask("Who published Grand Theft Auto: San Andreas?");

QUESTION: Who published Grand Theft Auto: San Andreas?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor

[STEP 1 - retrieve_game] query = 'Who published Grand Theft Auto: San Andreas?'


   found: Grand Theft Auto: San Andreas (PlayStation 2, 2004)  distance=0.123
   found: Gran Turismo (PlayStation 1, 1997)  distance=0.210
   found: Gran Turismo 5 (PlayStation 3, 2010)  distance=0.213
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor

[STEP 2 - evaluate_retrieval] question = 'Who published Grand Theft Auto: San Andreas?'


   useful=True  confidence=high
   reason: The document explicitly states that 'Rockstar Games' is the publisher of 'Grand Theft Auto: San Andreas'. This directly answers the query regarding who published the game.
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

--- REPORT ---
Tools used, in order: retrieve_game -> evaluate_retrieval
Web search needed: False

**Answer:** Grand Theft Auto: San Andreas was published by Rockstar Games.  
**Source:** Internal database: Grand Theft Auto: San Andreas  
**Confidence:** high, the document explicitly states the publisher.



### Conversation memory
This follow-up just says *"it"*. The agent only understands it because it remembers the previous question in the same session (`demo`).

In [13]:
ask("What platform was it released on, and in which year?");

QUESTION: What platform was it released on, and in which year?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor

[STEP 1 - retrieve_game] query = 'What platform was Grand Theft Auto: San Andreas released on, and in which year?'


   found: Grand Theft Auto: San Andreas (PlayStation 2, 2004)  distance=0.126
   found: Gran Turismo (PlayStation 1, 1997)  distance=0.205
   found: Gran Turismo 5 (PlayStation 3, 2010)  distance=0.209
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor

[STEP 2 - evaluate_retrieval] question = 'What platform was Grand Theft Auto: San Andreas released on, and in which year?'


   useful=True  confidence=high
   reason: The document provides both the platform ('PlayStation 2') and the year of release ('2004') for Grand Theft Auto: San Andreas. Since it directly answers the query, it is sufficient to conclude that the documents are useful.
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

--- REPORT ---
Tools used, in order: retrieve_game -> evaluate_retrieval
Web search needed: False

**Answer:** Grand Theft Auto: San Andreas was released on PlayStation 2 in 2004.  
**Source:** Internal database: Grand Theft Auto: San Andreas  
**Confidence:** high, the document explicitly states both the platform and the year of release.



**Query 6 (needs the web).** Current news, which a static dataset can't know.

In [14]:
ask("What is Rockstar Games working on right now?");

QUESTION: What is Rockstar Games working on right now?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor

[STEP 1 - retrieve_game] query = 'What is Rockstar Games working on right now?'


   found: Grand Theft Auto: San Andreas (PlayStation 2, 2004)  distance=0.218
   found: Gran Turismo 5 (PlayStation 3, 2010)  distance=0.225
   found: Halo Infinite (Xbox Series X|S, 2021)  distance=0.238
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor

[STEP 2 - evaluate_retrieval] question = 'What is Rockstar Games working on right now?'


   useful=False  confidence=low
   reason: The document provided only contains information about a past game, 'Grand Theft Auto: San Andreas', released in 2004. It does not provide any current information about what Rockstar Games is currently working on. Therefore, it does not answer the query.
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor

[STEP 3 - game_web_search] question = 'What is Rockstar Games working on right now?'


   source: https://www.rockstargames.com/VI?utm_source=openai
   source: https://www.techradar.com/gaming/gta-6-digital-pre-loads-for-ps5-and-xbox-will-begin-9-days-early-and-require-a-whopping-212gb-of-free-space?utm_source=openai
   source: https://portalgta6.com/en/news/rockstar-next-game-after-gta-6-not-eight-years?utm_source=openai
   source: https://www.gamesradar.com/games/grand-theft-auto/the-day-gta-6-pre-loading-starts-unionizing-rockstar-developers-will-vote-on-the-future-of-the-legendary-studio-this-is-our-moment/?utm_source=openai
   source: https://www.axios.com/2026/11/18/gta-6-playstation-psn-servers?utm_source=openai
   source: https://www.gamesradar.com/games/grand-theft-auto/as-gta-6-finally-nears-release-analysts-estimate-rockstar-has-already-made-usd1-45-billion-helped-by-nearly-90-percent-of-fans-buying-the-controversial-usd100-ultimate-edition/?utm_source=openai
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

--- REPORT ---
Tools used, in order: retrieve_game -> evaluate_retrieval -> game_web_search
Web search needed: True

**Answer:** Rockstar Games has recently released Grand Theft Auto VI (GTA VI) as of November 19, 2026. They have not officially announced their next project, but executives have indicated that the technology and team developed for GTA VI will facilitate faster development of future titles.  
**Source:** Web: [Rockstar Games](https://www.rockstargames.com/VI?utm_source=openai), [TechRadar](https://www.techradar.com/gaming/gta-6-digital-pre-loads-for-ps5-and-xbox-will-begin-9-days-early-and-require-a-whopping-212gb-of-free-space?utm_source=openai), [Portal GTA 6](https://portalgta6.com/en/news/rockstar-next-game-after-gta-6-not-eight-years?utm_source=openai), [GamesRadar](https://www.gamesradar.com/games/grand-theft-auto/the-day-gta-6-pre-loading-starts-unionizing-rockstar-developers-

**Memory is kept per session.** Every question above went into the session `demo`, so the agent carried that history with it. A new session id starts empty.

In [15]:
print("Questions remembered in session 'demo':", len(agent.get_session_runs("demo")))
agent.memory.create_session("brand_new")  # a fresh, empty session
print("Questions remembered in session 'brand_new':", len(agent.get_session_runs("brand_new")))

Questions remembered in session 'demo': 6
Questions remembered in session 'brand_new': 0


### Summary
* Questions about the 15 local games were answered from the **vector database**, with no web search.
* Questions outside the dataset (Mortal Kombat X, Rockstar's current work) were judged **not useful** and fell back to **web search**, citing URLs.
* Every answer has a **source** and a **confidence** level.
* The agent remembers earlier questions **within a session**.